In [10]:
"""
Bespoke generative decoders for trial-averaged Bernoulli features

Implements (A) Beta Naive Bayes on averaged probabilities x_ij in (0,1)
and (B) Beta-Binomial Naive Bayes if you know trial count T (uses k_ij ≈ round(T*x_ij)).

Outputs:
- evidence matrix Phi (same shape as X): phi_ij = log p(x_ij|y=1) - log p(x_ij|y=0)
- generative score s_i = logit(pi) + sum_j phi_ij
- optional PLS reduction on Phi + logistic regression in reduced space

Designed for large p (e.g. 40k neurons). Uses chunking and vectorization.
"""

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression

from scipy.special import betaln, gammaln


# ============================================================
# NUMERICS
# ============================================================

def _clip01(x, eps=1e-6):
    return np.clip(x, eps, 1.0 - eps)

def _logit(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def _beta_logpdf(x, a, b):
    # x: (n, p), a,b: (p,)
    # returns (n, p)
    return (a - 1.0) * np.log(x) + (b - 1.0) * np.log(1.0 - x) - betaln(a, b)

def _betabinom_logpmf(k, T, a, b):
    # k: (n, p) ints, a,b: (p,)
    # log [ C(T,k) * B(k+a, T-k+b) / B(a,b) ]
    # returns (n,p)
    k = k.astype(np.int64)
    logC = gammaln(T + 1) - gammaln(k + 1) - gammaln(T - k + 1)
    return logC + betaln(k + a, (T - k) + b) - betaln(a, b)


# ============================================================
# ESTIMATION: mu_jc and shared kappa
# ============================================================

def estimate_mu_per_class(X, y, a0=0.5, b0=0.5):
    """
    X: (n, p) in (0,1)
    y: (n,) in {0,1}
    Returns mu0, mu1: (p,)
    Uses a Beta(a0,b0) pseudo-count shrinkage on the mean:
      mu = (sum x + a0) / (n_c + a0 + b0)
    """
    X = _clip01(X)
    y = y.astype(int)
    idx0 = (y == 0)
    idx1 = (y == 1)
    n0 = max(int(idx0.sum()), 1)
    n1 = max(int(idx1.sum()), 1)

    s0 = X[idx0].sum(axis=0) if idx0.any() else np.zeros(X.shape[1])
    s1 = X[idx1].sum(axis=0) if idx1.any() else np.zeros(X.shape[1])

    mu0 = (s0 + a0) / (n0 + a0 + b0)
    mu1 = (s1 + a0) / (n1 + a0 + b0)
    return _clip01(mu0), _clip01(mu1)

def estimate_shared_kappa(X, y, mu0, mu1, kappa_min=2.0, kappa_max=1e6):
    """
    Shared concentration kappa using a pooled method-of-moments:
      Var[ X | class c ] ≈ mu_c(1-mu_c) / (kappa + 1)
    We estimate average empirical variance across neurons and classes,
    and solve for kappa.
    """
    X = _clip01(X)
    y = y.astype(int)
    idx0 = (y == 0)
    idx1 = (y == 1)

    # Empirical variances per neuron within each class (ddof=1 if possible)
    def safe_var(A):
        if A.shape[0] <= 1:
            return np.zeros(A.shape[1])
        return A.var(axis=0, ddof=1)

    v0 = safe_var(X[idx0]) if idx0.any() else np.zeros(X.shape[1])
    v1 = safe_var(X[idx1]) if idx1.any() else np.zeros(X.shape[1])

    # Expected beta variance numerator per class
    num0 = mu0 * (1.0 - mu0)
    num1 = mu1 * (1.0 - mu1)

    # Pool across classes and neurons robustly
    num = 0.5 * (num0 + num1)
    den = 0.5 * (v0 + v1)

    # Avoid dividing by ~0 variances (super-stable neurons)
    mask = den > np.percentile(den, 10)  # keep the more informative 90%
    if mask.sum() < 10:
        mask = den > 0

    if mask.sum() == 0:
        return 50.0  # fallback

    num_m = float(np.mean(num[mask]))
    den_m = float(np.mean(den[mask]))

    # kappa ≈ num/var - 1
    kappa = (num_m / max(den_m, 1e-12)) - 1.0
    kappa = float(np.clip(kappa, kappa_min, kappa_max))
    return kappa


# ============================================================
# GENERATIVE DECODERS
# ============================================================

class BetaNaiveBayesEvidence:
    """
    x_ij | y=c ~ Beta(alpha_jc, beta_jc)
    alpha_jc = mu_jc * kappa, beta_jc = (1-mu_jc)*kappa
    Shared kappa across neurons and classes (estimated from data unless provided).
    """

    def __init__(self, a0=0.5, b0=0.5, kappa=None, eps=1e-6, chunk=4096):
        self.a0 = a0
        self.b0 = b0
        self.kappa = kappa
        self.eps = eps
        self.chunk = chunk

        # fitted
        self.mu0_ = None
        self.mu1_ = None
        self.alpha0_ = None
        self.beta0_ = None
        self.alpha1_ = None
        self.beta1_ = None
        self.pi_ = None

    def fit(self, X, y):
        X = _clip01(np.asarray(X, float), self.eps)
        y = np.asarray(y, int)

        self.pi_ = float(np.mean(y))
        mu0, mu1 = estimate_mu_per_class(X, y, self.a0, self.b0)
        self.mu0_, self.mu1_ = mu0, mu1

        kappa = self.kappa
        if kappa is None:
            kappa = estimate_shared_kappa(X, y, mu0, mu1)
        self.kappa = float(kappa)

        self.alpha0_ = _clip01(mu0, self.eps) * self.kappa
        self.beta0_  = (1.0 - _clip01(mu0, self.eps)) * self.kappa
        self.alpha1_ = _clip01(mu1, self.eps) * self.kappa
        self.beta1_  = (1.0 - _clip01(mu1, self.eps)) * self.kappa
        return self

    def evidence_matrix(self, X):
        """
        Phi_ij = log p(x_ij|y=1) - log p(x_ij|y=0)
        Returns Phi of shape (n, p). Chunked over neurons for memory.
        """
        X = _clip01(np.asarray(X, float), self.eps)
        n, p = X.shape
        Phi = np.empty((n, p), dtype=np.float32)

        for j0 in range(0, p, self.chunk):
            j1 = min(p, j0 + self.chunk)
            Xc = X[:, j0:j1]
            ll1 = _beta_logpdf(Xc, self.alpha1_[j0:j1], self.beta1_[j0:j1])
            ll0 = _beta_logpdf(Xc, self.alpha0_[j0:j1], self.beta0_[j0:j1])
            Phi[:, j0:j1] = (ll1 - ll0).astype(np.float32)
        return Phi

    def score(self, X):
        """
        s_i = logit(pi) + sum_j phi_ij
        """
        Phi = self.evidence_matrix(X)
        return _logit(self.pi_) + Phi.sum(axis=1)

    def predict_proba(self, X):
        s = self.score(X)
        p1 = 1.0 / (1.0 + np.exp(-s))
        return np.vstack([1 - p1, p1]).T

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X)[:, 1] >= thresh).astype(int)


class BetaBinomialNaiveBayesEvidence:
    """
    If you know T (trials per image), use k_ij | y=c ~ BetaBinomial(T, alpha_jc, beta_jc)
    We take k_ij ≈ round(T * x_ij) if only x_ij is available.
    Same mu/kappa parameterization for stability.
    """

    def __init__(self, T, a0=0.5, b0=0.5, kappa=None, eps=1e-6, chunk=4096):
        self.T = int(T)
        self.a0 = a0
        self.b0 = b0
        self.kappa = kappa
        self.eps = eps
        self.chunk = chunk

        self.mu0_ = None
        self.mu1_ = None
        self.alpha0_ = None
        self.beta0_ = None
        self.alpha1_ = None
        self.beta1_ = None
        self.pi_ = None

    def fit(self, X, y):
        X = _clip01(np.asarray(X, float), self.eps)
        y = np.asarray(y, int)

        self.pi_ = float(np.mean(y))
        mu0, mu1 = estimate_mu_per_class(X, y, self.a0, self.b0)
        self.mu0_, self.mu1_ = mu0, mu1

        kappa = self.kappa
        if kappa is None:
            kappa = estimate_shared_kappa(X, y, mu0, mu1)
        self.kappa = float(kappa)

        self.alpha0_ = _clip01(mu0, self.eps) * self.kappa
        self.beta0_  = (1.0 - _clip01(mu0, self.eps)) * self.kappa
        self.alpha1_ = _clip01(mu1, self.eps) * self.kappa
        self.beta1_  = (1.0 - _clip01(mu1, self.eps)) * self.kappa
        return self

    def evidence_matrix(self, X):
        X = _clip01(np.asarray(X, float), self.eps)
        n, p = X.shape
        Phi = np.empty((n, p), dtype=np.float32)

        k = np.rint(self.T * X).astype(np.int64)  # approximate counts
        k = np.clip(k, 0, self.T)

        for j0 in range(0, p, self.chunk):
            j1 = min(p, j0 + self.chunk)
            kc = k[:, j0:j1]
            ll1 = _betabinom_logpmf(kc, self.T, self.alpha1_[j0:j1], self.beta1_[j0:j1])
            ll0 = _betabinom_logpmf(kc, self.T, self.alpha0_[j0:j1], self.beta0_[j0:j1])
            Phi[:, j0:j1] = (ll1 - ll0).astype(np.float32)
        return Phi

    def score(self, X):
        Phi = self.evidence_matrix(X)
        return _logit(self.pi_) + Phi.sum(axis=1)

    def predict_proba(self, X):
        s = self.score(X)
        from scipy.special import expit
        p1 = expit(s)
        return np.vstack([1 - p1, p1]).T

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X)[:, 1] >= thresh).astype(int)


# ============================================================
# PLS ON EVIDENCE + LOGISTIC
# ============================================================

def fit_pls_logistic(Phi_train, y_train, Phi_test, n_components=5, C=1.0, max_iter=2000):
    """
    PLSRegression learns supervised components of Phi for y.
    Then logistic regression is fit on the PLS scores.

    Returns: (yhat_test, proba_test, pls, clf, Z_train, Z_test)
    """
    # PLS wants y as float column
    pls = PLSRegression(n_components=n_components, scale=False)
    pls.fit(Phi_train, y_train.astype(float).reshape(-1, 1))

    Z_train = pls.transform(Phi_train)
    Z_test  = pls.transform(Phi_test)

    clf = LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=max_iter)
    clf.fit(Z_train, y_train)

    proba = clf.predict_proba(Z_test)[:, 1]
    yhat = (proba >= 0.5).astype(int)
    return yhat, proba, pls, clf, Z_train, Z_test


# ============================================================
# DEMO: LOAD YOUR DATA + RUN TRUE vs PERM CV
# ============================================================

if __name__ == "__main__":
    # ----------------------------
    # LOAD YOUR DATA
    # ----------------------------
    VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
    NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'
    AREAS_PATH  = '/home/maria/ProjectionSort/data/brain_area.npy'

    vit   = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
    R     = np.load(NEURAL_PATH).T                                   # (images, neurons)  <-- should be in (0,1)
    areas = np.load(AREAS_PATH, allow_pickle=True)

    print("Images:", vit.shape[0])
    print("Neurons:", R.shape[1])

    # Animate / inanimate label (your heuristic)
    top1 = np.argmax(vit, axis=1)
    y_true = (top1 <= 397).astype(int)
    print("Animate fraction:", y_true.mean())

    # Make sure features are in (0,1)
    X = _clip01(R.astype(float), eps=1e-6)

    # ----------------------------
    # CHOOSE MODEL
    # ----------------------------
    # If you DO NOT know T, use Beta NB:
    model = BetaNaiveBayesEvidence(a0=0.5, b0=0.5, kappa=None, chunk=4096)

    # If you DO know T (e.g. T=20 trials per image), uncomment:
    model = BetaBinomialNaiveBayesEvidence(T=50, a0=0.5, b0=0.5, kappa=None, chunk=4096)

    # ----------------------------
    # CV EVAL: (1) pure generative sum of evidence, (2) PLS+logistic on Phi
    # ----------------------------
    skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=0)

    def run_cv(y, tag):
        acc_gen = []
        acc_pls = []
        kappas = []

        for fold, (tr, te) in enumerate(skf.split(X, y), 1):
            Xtr, Xte = X[tr], X[te]
            ytr, yte = y[tr], y[te]

            # ----------------------------------
            # CLEAN MODEL CONSTRUCTION
            # ----------------------------------
            if isinstance(model, BetaNaiveBayesEvidence):
                m = BetaNaiveBayesEvidence(
                    a0=model.a0,
                    b0=model.b0,
                    kappa=model.kappa,
                    eps=model.eps,
                    chunk=model.chunk
                )
            else:
                m = BetaBinomialNaiveBayesEvidence(
                    T=model.T,
                    a0=model.a0,
                    b0=model.b0,
                    kappa=model.kappa,
                    eps=model.eps,
                    chunk=model.chunk
                )

            # ----------------------------------
            # FIT GENERATIVE MODEL
            # ----------------------------------
            m.fit(Xtr, ytr)
            kappas.append(m.kappa)

            # (1) Pure generative classifier
            proba_gen = m.predict_proba(Xte)[:, 1]
            yhat_gen = (proba_gen >= 0.5).astype(int)
            acc_gen.append((yhat_gen == yte).mean())

            # ----------------------------------
            # Evidence features for PLS
            # ----------------------------------
            Phi_tr = m.evidence_matrix(Xtr)
            Phi_te = m.evidence_matrix(Xte)

            # (2) PLS + logistic in evidence space
            ncomp = min(5, Phi_tr.shape[0] - 1)
            ncomp=20
            yhat_pls, proba_pls, *_ = fit_pls_logistic(
                Phi_tr, ytr, Phi_te,
                n_components=ncomp,
                C=1.0
            )
            acc_pls.append((yhat_pls == yte).mean())

            print(f"[{tag}] fold {fold}: "
                f"acc_gen={acc_gen[-1]:.3f}, "
                f"acc_pls={acc_pls[-1]:.3f}, "
                f"kappa={m.kappa:.2f}")

        print(f"\n[{tag}] Generative-only mean acc: {np.mean(acc_gen):.3f} ± {np.std(acc_gen):.3f}")
        print(f"[{tag}] PLS+logistic mean acc:   {np.mean(acc_pls):.3f} ± {np.std(acc_pls):.3f}")
        print(f"[{tag}] kappa (mean±std):       {np.mean(kappas):.2f} ± {np.std(kappas):.2f}\n")


    # TRUE labels
    run_cv(y_true, "TRUE")

    # PERM labels (sanity check)
    rng = np.random.default_rng(0)
    y_perm = y_true.copy()
    rng.shuffle(y_perm)
    run_cv(y_perm, "PERM")

    # ----------------------------
    # FIT ON ALL DATA ONCE + EXPORT Phi FOR DOWNSTREAM ANALYSIS
    # ----------------------------
    model.fit(X, y_true)
    Phi = model.evidence_matrix(X)  # (images, neurons)
    scores = _logit(model.pi_) + Phi.sum(axis=1)

    print("Final fit: kappa =", model.kappa)
    print("Phi shape:", Phi.shape)
    print("Score stats: mean=", float(scores.mean()), "std=", float(scores.std()))

    # Save evidence features if you want:
    # np.save("/home/maria/ProjectionSort/data/evidence_phi.npy", Phi)
    # np.save("/home/maria/ProjectionSort/data/evidence_score.npy", scores)


Images: 118
Neurons: 39209
Animate fraction: 0.5338983050847458
[TRUE] fold 1: acc_gen=0.500, acc_pls=0.600, kappa=16.47
[TRUE] fold 2: acc_gen=0.650, acc_pls=0.700, kappa=16.14
[TRUE] fold 3: acc_gen=0.650, acc_pls=0.650, kappa=15.75
[TRUE] fold 4: acc_gen=0.800, acc_pls=0.800, kappa=16.25
[TRUE] fold 5: acc_gen=0.632, acc_pls=0.632, kappa=16.66
[TRUE] fold 6: acc_gen=0.684, acc_pls=0.632, kappa=16.29

[TRUE] Generative-only mean acc: 0.653 ± 0.088
[TRUE] PLS+logistic mean acc:   0.669 ± 0.066
[TRUE] kappa (mean±std):       16.26 ± 0.28

[PERM] fold 1: acc_gen=0.450, acc_pls=0.500, kappa=16.31
[PERM] fold 2: acc_gen=0.550, acc_pls=0.450, kappa=15.78
[PERM] fold 3: acc_gen=0.500, acc_pls=0.450, kappa=15.90
[PERM] fold 4: acc_gen=0.550, acc_pls=0.450, kappa=16.22
[PERM] fold 5: acc_gen=0.526, acc_pls=0.526, kappa=16.28
[PERM] fold 6: acc_gen=0.263, acc_pls=0.474, kappa=16.03

[PERM] Generative-only mean acc: 0.473 ± 0.100
[PERM] PLS+logistic mean acc:   0.475 ± 0.029
[PERM] kappa (mean±

In [7]:
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# ----------------------------
# Data already loaded:
# R : (images, neurons)
# y_true : (images,)
# ----------------------------

X = R.astype(float)
y = y_true.astype(int)

skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=0)

acc = []

for fold, (tr, te) in enumerate(skf.split(X, y), 1):
    Xtr, Xte = X[tr], X[te]
    ytr, yte = y[tr], y[te]

    # Standardize neurons using training fold only
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)

    # PLS directly on raw neural data
    ncomp = min(5, Xtr.shape[0] - 1)
    ncomp=20
    pls = PLSRegression(n_components=ncomp)
    pls.fit(Xtr, ytr.reshape(-1, 1))

    Ztr = pls.transform(Xtr)
    Zte = pls.transform(Xte)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(Ztr, ytr)

    acc_fold = clf.score(Zte, yte)
    acc.append(acc_fold)

    print(f"[RAW PLS] fold {fold}: acc = {acc_fold:.3f}")

print("\n[RAW PLS] mean acc:", np.mean(acc), "±", np.std(acc))


[RAW PLS] fold 1: acc = 0.550
[RAW PLS] fold 2: acc = 0.700
[RAW PLS] fold 3: acc = 0.650
[RAW PLS] fold 4: acc = 0.750
[RAW PLS] fold 5: acc = 0.737
[RAW PLS] fold 6: acc = 0.684

[RAW PLS] mean acc: 0.6785087719298245 ± 0.06625130300059537


In [11]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy.special import expit

# ----------------------------------------
# X : (n_images, n_neurons) in (0,1)
# y_true : (n_images,)
# model = BetaBinomialNaiveBayesEvidence(T=50, ...)
# ----------------------------------------

def loo_beta_binomial_cv(X, y, T=50, a0=0.5, b0=0.5, kappa=None, eps=1e-6):
    n = X.shape[0]
    y = y.astype(int)

    yhat = np.zeros(n, dtype=int)
    proba = np.zeros(n, dtype=float)
    kappas = np.zeros(n)

    for i in range(n):
        # Leave i out
        mask = np.ones(n, dtype=bool)
        mask[i] = False

        Xtr, Xte = X[mask], X[~mask]
        ytr, yte = y[mask], y[~mask]

        # Fit model
        m = BetaBinomialNaiveBayesEvidence(
            T=T, a0=a0, b0=b0, kappa=kappa, eps=eps, chunk=4096
        )
        m.fit(Xtr, ytr)
        kappas[i] = m.kappa

        # Predict left-out sample
        s = m.score(Xte)[0]
        p = expit(s)
        proba[i] = p
        yhat[i] = int(p >= 0.5)

        if (i+1) % 10 == 0 or i == n-1:
            print(f"LOO {i+1}/{n}  kappa={m.kappa:.2f}  p={p:.3f}")

    acc = accuracy_score(y, yhat)
    auc = roc_auc_score(y, proba)

    print("\n=== LOO Beta–Binomial Results ===")
    print("Accuracy:", acc)
    print("ROC-AUC :", auc)
    print("kappa   :", np.mean(kappas), "±", np.std(kappas))

    return yhat, proba, kappas


yhat_loo, proba_loo, kappas_loo = loo_beta_binomial_cv(
    X, y_true, T=50, a0=0.5, b0=0.5
)


LOO 10/118  kappa=15.88  p=1.000
LOO 20/118  kappa=16.03  p=1.000
LOO 30/118  kappa=15.93  p=1.000
LOO 40/118  kappa=15.89  p=1.000
LOO 50/118  kappa=15.83  p=0.000
LOO 60/118  kappa=15.83  p=0.000
LOO 70/118  kappa=16.00  p=0.000
LOO 80/118  kappa=15.82  p=0.000
LOO 90/118  kappa=15.84  p=1.000
LOO 100/118  kappa=15.87  p=0.000
LOO 110/118  kappa=15.89  p=1.000
LOO 118/118  kappa=15.91  p=1.000

=== LOO Beta–Binomial Results ===
Accuracy: 0.6694915254237288
ROC-AUC : 0.7331890331890332
kappa   : 15.859296245333617 ± 0.045315688199316986


In [18]:
import numpy as np
from scipy.special import expit

# ============================================================
# PATHS
# ============================================================
VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'

# ============================================================
# LOAD DATA
# ============================================================
vit = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']
R   = np.load(NEURAL_PATH).T

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

top1 = np.argmax(vit, axis=1)
y_true = (top1 <= 397).astype(int)

X = np.clip(R.astype(float), 1e-6, 1 - 1e-6)

# ============================================================
# FISHER GEOMETRY FUNCTIONS
# ============================================================

def fisher_I_t_from_scores(scores, t=1.0):
    s = scores / (np.std(scores) + 1e-12)
    p = expit(t * s)
    return float(np.sum(p * (1 - p) * (s ** 2)))

def fisher_contribs_t(scores, t=1.0):
    s = scores / (np.std(scores) + 1e-12)
    p = expit(t * s)
    return p * (1 - p) * (s ** 2)

def summarize_fisher_contribs(contrib):
    total = contrib.sum()
    top5 = np.sort(contrib)[-5:].sum()
    top10 = np.sort(contrib)[-len(contrib)//10:].sum()
    return dict(total_I=float(total),
                top5_frac=float(top5/total),
                top10pct_frac=float(top10/total))

def bootstrap_fisher_I(scores, B=2000):
    rng = np.random.default_rng(0)
    out = np.zeros(B)
    for i in range(B):
        s = scores[rng.integers(0, len(scores), len(scores))]
        out[i] = fisher_I_t_from_scores(s)
    return out

# ============================================================
# LOO BETA–BINOMIAL GENERATIVE SCORES
# ============================================================

def loo_scores(X, y, T=50):
     # your existing class

    n = len(y)
    scores = np.zeros(n)

    for i in range(n):
        mask = np.ones(n, bool)
        mask[i] = False

        m = BetaBinomialNaiveBayesEvidence(T=T, chunk=4096)
        m.fit(X[mask], y[mask])
        scores[i] = m.score(X[~mask])[0]

        if (i+1) % 10 == 0:
            print(f"LOO {i+1}/{n}")

    return scores

print("\nRunning LOO for TRUE labels...")
scores_true = loo_scores(X, y_true)

print("\nRunning LOO for PERM labels...")
rng = np.random.default_rng(0)
y_perm = y_true.copy()
rng.shuffle(y_perm)
scores_perm = loo_scores(X, y_perm)

# ============================================================
# FISHER GEOMETRY TEST
# ============================================================

print("\n--- Fisher diagnostics ---")

contrib = fisher_contribs_t(scores_true)
print("TRUE contrib summary:", summarize_fisher_contribs(contrib))

I_true = bootstrap_fisher_I(scores_true)
I_perm = bootstrap_fisher_I(scores_perm)

print("TRUE Fisher percentiles:", np.percentile(I_true, [2.5, 50, 97.5]))
print("PERM Fisher percentiles:", np.percentile(I_perm, [2.5, 50, 97.5]))
print("Frac(PERM ≥ TRUE median):", np.mean(I_perm >= np.median(I_true)))


Images: 118
Neurons: 39209

Running LOO for TRUE labels...
LOO 10/118
LOO 20/118
LOO 30/118
LOO 40/118
LOO 50/118
LOO 60/118
LOO 70/118
LOO 80/118
LOO 90/118
LOO 100/118
LOO 110/118

Running LOO for PERM labels...
LOO 10/118
LOO 20/118
LOO 30/118
LOO 40/118
LOO 50/118
LOO 60/118
LOO 70/118
LOO 80/118
LOO 90/118
LOO 100/118
LOO 110/118

--- Fisher diagnostics ---
TRUE contrib summary: {'total_I': 18.32508033106034, 'top5_frac': 0.11673914097396047, 'top10pct_frac': 0.26507267998605205}
TRUE Fisher percentiles: [17.32287523 18.4516372  19.52440612]
PERM Fisher percentiles: [15.04413003 17.69822426 20.16940799]
Frac(PERM ≥ TRUE median): 0.307


In [19]:
from sklearn.model_selection import LeaveOneOut

def run_area_loo(X, y, areas, model, min_neurons=50, tag="TRUE"):
    """
    X: (images, neurons)
    y: (images,)
    areas: (neurons,)
    """

    uniq_areas = np.unique(areas)
    loo = LeaveOneOut()

    results = {}

    for area in uniq_areas:
        idx = np.where(areas == area)[0]
        if len(idx) < min_neurons:
            continue

        Xa = X[:, idx]
        acc_gen, acc_pls = [], []
        kappas = []

        print(f"\n=== Area {area} | neurons = {Xa.shape[1]} ===")

        for fold, (tr, te) in enumerate(loo.split(Xa), 1):
            Xtr, Xte = Xa[tr], Xa[te]
            ytr, yte = y[tr], y[te]

            # fresh model each fold
            if isinstance(model, BetaNaiveBayesEvidence):
                m = BetaNaiveBayesEvidence(
                    a0=model.a0, b0=model.b0,
                    kappa=model.kappa, eps=model.eps,
                    chunk=model.chunk
                )
            else:
                m = BetaBinomialNaiveBayesEvidence(
                    T=model.T,
                    a0=model.a0, b0=model.b0,
                    kappa=model.kappa, eps=model.eps,
                    chunk=model.chunk
                )

            m.fit(Xtr, ytr)
            kappas.append(m.kappa)

            # generative only
            p_gen = m.predict_proba(Xte)[0,1]
            acc_gen.append(int((p_gen >= 0.5) == yte[0]))

            # PLS + logistic
            Phi_tr = m.evidence_matrix(Xtr)
            Phi_te = m.evidence_matrix(Xte)

            ncomp = min(10, Phi_tr.shape[0]-1)
            yhat, *_ = fit_pls_logistic(Phi_tr, ytr, Phi_te, n_components=ncomp)
            acc_pls.append(int(yhat[0] == yte[0]))

        results[area] = {
            "acc_gen": np.mean(acc_gen),
            "acc_pls": np.mean(acc_pls),
            "kappa": np.mean(kappas),
            "n_neurons": len(idx)
        }

        print(f"[{tag}] Area {area}: "
              f"Gen={np.mean(acc_gen):.3f} | "
              f"PLS={np.mean(acc_pls):.3f} | "
              f"kappa={np.mean(kappas):.2f}")

    return results

print("\n\n===== AREA-WISE TRUE LABELS =====")
res_true = run_area_loo(X, y_true, areas, model, tag="TRUE")

rng = np.random.default_rng(0)
y_perm = y_true.copy()
rng.shuffle(y_perm)

print("\n\n===== AREA-WISE PERMUTED LABELS =====")
res_perm = run_area_loo(X, y_perm, areas, model, tag="PERM")




===== AREA-WISE TRUE LABELS =====

=== Area VISal | neurons = 4249 ===
[TRUE] Area VISal: Gen=0.669 | PLS=0.678 | kappa=15.84

=== Area VISam | neurons = 2040 ===
[TRUE] Area VISam: Gen=0.695 | PLS=0.695 | kappa=15.84

=== Area VISl | neurons = 8323 ===
[TRUE] Area VISl: Gen=0.669 | PLS=0.653 | kappa=15.84

=== Area VISp | neurons = 14382 ===
[TRUE] Area VISp: Gen=0.669 | PLS=0.712 | kappa=15.84

=== Area VISpm | neurons = 4771 ===
[TRUE] Area VISpm: Gen=0.678 | PLS=0.686 | kappa=15.84

=== Area VISrl | neurons = 5444 ===
[TRUE] Area VISrl: Gen=0.678 | PLS=0.661 | kappa=15.84


===== AREA-WISE PERMUTED LABELS =====

=== Area VISal | neurons = 4249 ===
[PERM] Area VISal: Gen=0.534 | PLS=0.475 | kappa=15.84

=== Area VISam | neurons = 2040 ===
[PERM] Area VISam: Gen=0.424 | PLS=0.458 | kappa=15.84

=== Area VISl | neurons = 8323 ===
[PERM] Area VISl: Gen=0.500 | PLS=0.551 | kappa=15.84

=== Area VISp | neurons = 14382 ===
[PERM] Area VISp: Gen=0.500 | PLS=0.458 | kappa=15.84

=== Area 